# Indie Comic Pipeline - Google Colab Edition
This notebook runs the Indie Comic Pipeline with free GPU acceleration for SDXL (5-10 seconds per image instead of 2 hours on CPU).

### How to run:
1. In the Colab menu, go to **Runtime > Change runtime type** and select **T4 GPU** (or any available GPU).
2. Upload your `indie_comic_pipeline` folder to this session (or mount your Google Drive).
3. Run the cells below in order.

### Step 1: Install Dependencies
This installs the required libraries including PyTorch with GPU compatibility, diffusers, accelerate, and langchain.

In [ ]:
!pip install diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python pillow scikit-image peft

### Step 2: Install and Start Ollama
This downloads the Ollama binary, launches the background daemon inside Colab, and pulls the `llama3.2` model.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama serve in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Give it 5 seconds to wake up

# Pull Llama 3.2 model
!ollama pull llama3.2

### Step 3: Configure Settings for Colab GPU
We will rewrite/update `config/settings.yaml` to use `cuda` (GPU device) and ensure standard SDXL resolution.

In [ ]:
import yaml

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for GPU execution of SDXL, SD v1.5 and LoRA
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['models']['sd15']['device'] = 'cuda'
settings['models']['sd15']['name'] = 'runwayml/stable-diffusion-v1-5'
settings['models']['lora']['name'] = 'artificialguybr/LineAniRedmond-LinearMangaSDXL-V2'
settings['models']['lora']['trigger_words'] = 'LineAniAF, lineart'
settings['generation']['default_size']['width'] = 1024
settings['generation']['default_size']['height'] = 1024
settings['generation']['inference_steps'] = 30

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)

### Step 4: Run LangChain Extraction Pipeline
Change the arguments below to customize the character and story setting.

In [ ]:
# Customize character and based story setting here:
CHARACTER = "Spiderman"
STORY_SETTING = "Cyberpunk"

!python langchain_code/character_extractor.py "{CHARACTER}"
!python langchain_code/story_extractor.py "{STORY_SETTING}"
!python langchain_code/fusion_engine.py

### Step 4.5: Run Emotion Recognition (ERC) Engine
Processes dialogue and story contexts to extract emotions, maps expressions to visual prompts, and prepares annotated layout instructions.

In [ ]:
!python langchain_code/emotion_recognition_engine.py

### Step 5: Run SDXL Image Generation Pipeline
This loads SDXL on GPU, renders the character reference and the 4 story components, and saves them.

In [ ]:
%cd sdxl_code
!python run_sdxl_pipeline.py
%cd ..

### Step 5.1: Run Stable Diffusion v1.5 Image Generation Pipeline
As an alternative to SDXL, this loads Stable Diffusion v1.5 on GPU, renders the character reference and panel components, and saves them.

In [ ]:
%cd sd15_code
!python run_sd15_pipeline.py
%cd ..

### Step 5.2: Run SDXL + LoRA Image Generation Pipeline
This loads SDXL with the Lineart LoRA on GPU, renders the character reference and panel components with enhanced artistic styling, and saves them.

In [ ]:
%cd lora_code
!python run_lora_pipeline.py
%cd ..

### Step 5.3: Run SDXL Emotion-Aware Comic Panel Generation
Generates individual comic panels and compiles page layouts using the base SDXL model.

In [ ]:
%cd sdxl_code
!python generate_panels.py --page 1
%cd ..

### Step 5.4: Run Stable Diffusion v1.5 Emotion-Aware Comic Panel Generation
Generates individual comic panels and compiles page layouts using SD 1.5 + LoRA styling.

In [ ]:
%cd sd15_code
!python generate_panels.py --page 1
%cd ..

### Step 5.5: Run SDXL + LoRA Emotion-Aware Comic Panel Generation
Generates individual comic panels and compiles page layouts using SDXL + Lineart LoRA model styling.

In [ ]:
%cd lora_code
!python generate_panels.py --page 1
%cd ..

### Step 5.6: Run Model Evaluation and Benchmarking Matrix
Compare Stable Diffusion v1.5, Stable Diffusion XL (Base), and SDXL with a Lineart LoRA side-by-side on CLIP Similarity Score, FID Score, and Inference speed.

In [ ]:
!python matrix_evaluation_zone/model_matrix_bench.py

### Step 5.7: Read the 10-Page Multiverse Crossover Script
This displays the full 10-page crossover storyboard script, including plot progression, character personality adaptation, panel descriptions, and dialogue/captions.

In [ ]:
import json

with open('outputs/fusion/fusion_complete.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

fusion = data.get('fusion', {})
personality = data.get('personality', {})
setting = data.get('setting', {})

print("=" * 80)
print(f"MULTIVERSE CROSSOVER EVENT: {personality.get('character_name')} IN {setting.get('story_name')}")
print(f"Theme of the World: {setting.get('theme', '')}")
print(f"Dialogue Style & Tone: {setting.get('dialogue_style_and_tone', '')}")
print(f"Cinematographic Visual Style: {setting.get('cinematographic_visual_style', '')}")
print(f"Adaptation Style: {fusion.get('character_visual_looks', '')}")
print(f"Adaptation Summary: {fusion.get('multiverse_adaptation_summary', '')}")
print("=" * 80)

pages = fusion.get('storyboard_10_pages', [])
for p in pages:
    print(f"\n=== PAGE {p.get('page_number')}: {p.get('location')} ===")
    print(f"Narrative Progression: {p.get('narrative_progression')}")
    print(f"Personality/Mood Shift: {p.get('personality_state')}")
    if p.get('side_characters_present'):
        print(f"Side Characters: {', '.join(p.get('side_characters_present'))}")
    print("Panels Breakdown:")
    for pb in p.get('panels_breakdown', []):
        print(f"  - {pb}")
    print("Dialogue & Captions:")
    for dc in p.get('dialogue_and_captions', []):
        print(f"  - {dc}")
    print("-" * 80)

### Step 6: View the Generated Assets (Visual Components & Comic Panels)
Displays the generated character reference profile, compiled asset sheet, and any generated emotion-aware comic panel layouts.

In [ ]:
from IPython.display import Image, display
import os
import glob

print("=== Character Reference Profile ===")
if os.path.exists("outputs/characters/character_reference.png"):
    display(Image("outputs/characters/character_reference.png"))
else:
    print("Not generated yet.")

print("\n=== Persistent Visual Assets Sheet (Dynamic Grid) ===")
if os.path.exists("outputs/comics/component_sheet_grid_2x2.png"):
    display(Image("outputs/comics/component_sheet_grid_2x2.png"))
else:
    print("Not generated yet.")

print("\n=== Emotion-Aware Comic Panel Layouts ===")
layouts = glob.glob("outputs/comics/page_*_layout_*_grid.png")
if layouts:
    for layout in sorted(layouts):
        print(f"Displaying: {layout}")
        display(Image(layout))
else:
    print("No panel layouts generated yet. Run Step 5.3, 5.4, or 5.5 first.")

### Step 5.8: Generate Custom Doodle Storyboard (Optional Test)
Runs the `generate_doodle_panels.py` script to generate 8 test panels from a custom story JSON and compiles them into a grid layout.

In [ ]:
!python generate_doodle_panels.py

### Step 6.1: View Custom Doodle Storyboard Layout
Displays the compiled 4x2 layout grid for the custom test story.

In [ ]:
print("=== Custom Doodle Storyboard Layout Grid ===")
if os.path.exists("outputs/comics/doodle_story_layout_grid.png"):
    display(Image("outputs/comics/doodle_story_layout_grid.png"))
else:
    print("Doodle storyboard not generated yet. Run Step 5.8 first.")